In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql2025;TrustServerCertificate=True;Integrated Security=True"

In [ ]:
SELECT @@VERSION

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='RCDemo')BEGIN
    ALTER DATABASE RCDemo SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE RCDemo;
END
GO
CREATE DATABASE RCDemo
GO
USE RCDemo
GO
create master key encryption by password = 'MyTest!Mast3rP4ss'
GO
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

In [ ]:
CREATE EXTERNAL MODEL ollama
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
SELECT AI_GENERATE_EMBEDDINGS('BDC is the best!',ollama)

In [ ]:
CREATE TABLE RandomTexts (
    RandomText NVARCHAR(max),
    embeddings VECTOR(768)
);

In [ ]:
DECLARE @response NVARCHAR(max)
DECLARE @prompt NVARCHAR(max) = N'Write a 2 page essay on the airspeed velocity of unladden swallows'
DECLARE @model NVARCHAR(250) = N'mistral'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;
TRUNCATE TABLE RandomTexts
INSERT INTO RandomTexts (RandomText)
SELECT value
FROM   Openjson(Json_query(@response, '$.result')) A
WHERE  [key] = 'response' 
SELECT * FROM RandomTexts

In [ ]:
UPDATE RandomTexts SET embeddings = AI_GENERATE_EMBEDDINGS(RandomText,ollama)

In [ ]:
DBCC TRACESTATUS

In [ ]:
CREATE VECTOR INDEX vec_idx ON RandomTexts([embeddings])
WITH (
    metric = 'cosine',
    type = 'diskann',
    maxdop = 8
)

In [ ]:
ALTER DATABASE SCOPED CONFIGURATION SET PREVIEW_FEATURES = ON

In [ ]:
CREATE VECTOR INDEX vec_idx ON RandomTexts([embeddings])
WITH (
    metric = 'cosine',
    type = 'diskann',
    maxdop = 8
)

In [ ]:
ALTER TABLE RandomTexts
ADD [ID] [int] IDENTITY(1,1) NOT NULL

In [ ]:
ALTER TABLE RandomTexts
ADD CONSTRAINT PK_a_Id PRIMARY KEY (Id)

In [ ]:
CREATE VECTOR INDEX vec_idx ON RandomTexts([embeddings])
WITH (
    metric = 'cosine',
    type = 'diskann',
    maxdop = 8
)

In [ ]:
UPDATE RandomTexts SET RandomText = ''

In [ ]:
SELECT RandomText FROM RandomTexts

In [ ]:
SELECT c.* FROM RandomTexts t
CROSS APPLY
   AI_GENERATE_CHUNKS(source = t.RandomText, chunk_type = FIXED, chunk_size = 500, 
   enable_chunk_set_id = 1) c